In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [1]:
import os
import numpy as np
import pandas as pd

BASE_DIR = "/content/drive/MyDrive/Mining of massive dataset/preprocessing_output"
HOL_PATH = os.path.join(BASE_DIR, "holidays_2018.csv")
WEA_PATH = os.path.join(BASE_DIR, "weather_2018.csv")

In [15]:
TIME_BINS_PATH = "/content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2018/time_bins.npy"
time_bins = np.load(TIME_BINS_PATH)

time_bins = pd.to_datetime(time_bins)
print("time_bins len:", len(time_bins), "| range:", time_bins.min(), "->", time_bins.max())

time_bins len: 17519 | range: 2018-01-01 00:00:00 -> 2018-12-31 23:30:00


In [3]:
hol = pd.read_csv(HOL_PATH)
wea = pd.read_csv(WEA_PATH)

hol["time"] = pd.to_datetime(hol["time"], errors="coerce")
wea["time"] = pd.to_datetime(wea["time"], errors="coerce")

hol = hol.dropna(subset=["time"]).sort_values("time")
wea = wea.dropna(subset=["time"]).sort_values("time")

In [4]:
print("hol rows:", len(hol), "time range:", hol["time"].min(), "->", hol["time"].max())
print("wea rows:", len(wea), "time range:", wea["time"].min(), "->", wea["time"].max())
print("hol cols:", hol.columns.tolist())
print("wea cols:", wea.columns.tolist())

hol rows: 17519 time range: 2018-01-01 00:00:00 -> 2018-12-31 23:00:00
wea rows: 17519 time range: 2018-01-01 00:00:00 -> 2018-12-31 23:00:00
hol cols: ['time', 'Is_Legal_Holiday', 'Is_School_Recess', 'Is_Event_Festival', 'Is_Early_Dismissal']
wea cols: ['time', 'temperature_2m', 'cloudcover', 'windspeed_10m', 'Rain_None', 'Rain_Light', 'Rain_Moderate', 'Rain_Heavy']


### merge holiday + weather

In [16]:
ctx = pd.DataFrame({"time": time_bins})

ctx = ctx.merge(hol, on="time", how="left").merge(wea, on="time", how="left")

In [17]:
ctx = ctx.fillna(0)

print("ctx shape:", ctx.shape)
ctx.head()

ctx shape: (17519, 12)


,time,Is_Legal_Holiday,Is_School_Recess,Is_Event_Festival,Is_Early_Dismissal,temperature_2m,cloudcover,windspeed_10m,Rain_None,Rain_Light,Rain_Moderate,Rain_Heavy
0,2018-01-01 00:00:00,1.0,0.0,0.0,0.0,0.136364,0.0,0.200445,1.0,0.0,0.0,0.0
1,2018-01-01 00:30:00,1.0,0.0,0.0,0.0,0.132576,0.0,0.208241,1.0,0.0,0.0,0.0
2,2018-01-01 01:00:00,1.0,0.0,0.0,0.0,0.128788,0.0,0.216036,1.0,0.0,0.0,0.0
3,2018-01-01 01:30:00,1.0,0.0,0.0,0.0,0.125947,0.0,0.240535,1.0,0.0,0.0,0.0
4,2018-01-01 02:00:00,1.0,0.0,0.0,0.0,0.123106,0.0,0.265033,1.0,0.0,0.0,0.0


In [18]:
holiday_cols = ["Is_Legal_Holiday", "Is_School_Recess", "Is_Event_Festival", "Is_Early_Dismissal"]
weather_cont_cols = ["temperature_2m", "cloudcover", "windspeed_10m"]
weather_oh_cols = ["Rain_None", "Rain_Light", "Rain_Moderate", "Rain_Heavy"]

In [19]:
for c in holiday_cols + weather_cont_cols + weather_oh_cols:
    if c not in ctx.columns:
        print("Missing col:", c)

In [20]:
CTX = np.concatenate([
    ctx[holiday_cols].to_numpy(np.float32),
    ctx[weather_cont_cols].to_numpy(np.float32),
    ctx[weather_oh_cols].to_numpy(np.float32),
], axis=1).astype(np.float32)

In [21]:
print("CTX shape [T,K]:", CTX.shape)

CTX shape [T,K]: (17519, 11)


In [12]:
VOL_TENSOR_PATH = os.path.join("/content/drive/MyDrive/Mining of massive dataset/preprocessing_output/yellow_data/yellow_2018/", "taxi_volume_4d_tensor_final.npy")
vol_tensor = np.load(VOL_TENSOR_PATH)

In [22]:
print("volume tensor T:", vol_tensor.shape[0])
print("CTX T:", CTX.shape[0])

volume tensor T: 17519
CTX T: 17519


In [24]:
CTX_PATH = os.path.join(BASE_DIR, "CTX_2018_aligned.npy")
np.save(CTX_PATH, CTX)
print("Saved CTX to:", CTX_PATH)

Saved CTX to: /content/drive/MyDrive/Mining of massive dataset/preprocessing_output/CTX_2018_aligned.npy


## Dataset for tensor_4d.npy + CTX_2018.npy

In [ ]:
class TLCdataset(Dataset):
    def __init__(self, raw_npy, ctx, seq_len=8, pred_len=1, crop_size=9, min_val=None, max_val=None):
        self.ctx = torch.FloatTensor(ctx)
        raw_data = raw_npy

        self.min_val = min_val
        self.max_val = max_val
        normalized_data = (raw_data - self.min_val) / (self.max_val - self.min_val + 1e-7)
        self.data = torch.FloatTensor(normalized_data)

        self.T, self.C, self.H, self.W = self.data.shape
        assert self.ctx.shape[0] == self.T, "CTX T must match raw_npy T"

        self.seq_len = seq_len
        self.pred_len = pred_len
        self.crop_size = crop_size

        self.valid_times = self.T - self.seq_len - self.pred_len + 1
        self.h_steps = self.H - self.crop_size + 1
        self.w_steps = self.W - self.crop_size + 1
        self.patches_per_time = self.h_steps * self.w_steps

    def __len__(self):
        return self.valid_times * self.patches_per_time

    def __getitem__(self, idx):
        time_idx = idx // self.patches_per_time
        patch_idx = idx % self.patches_per_time

        h_start = patch_idx // self.w_steps
        w_start = patch_idx % self.w_steps

        X_full = self.data[time_idx: time_idx + self.seq_len]
        Y_full = self.data[time_idx + self.seq_len: time_idx + self.seq_len + self.pred_len]

        X_crop = X_full[:, :, h_start:h_start + self.crop_size, w_start:w_start + self.crop_size]
        center_h = h_start + self.crop_size // 2
        center_w = w_start + self.crop_size // 2

        Y_center = Y_full[:, 0, center_h, center_w]

        CTX_seq = self.ctx[time_idx: time_idx + self.seq_len]
        return X_crop, CTX_seq, Y_center